In [ ]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(name)s — %(levelname)s — %(message)s',
)

# Covariance-Based Amplitude SNR Filter — Simulation Validation

Tests the proposed z_amplitude statistic as a replacement for the current three hard gates:
- `MEDIAN_GATE_THRESHOLD = 2.0 pe` (Stage 1)
- `MIN_PHOTON_THRESHOLD = 50.0 pe` (Stage 2)
- `MAX_CHI_SQUARED = 3.0` (Stage 2)

**Method:** simulate Bayer images with `gen_camera_image_stack` at known photon levels.
Feed the WLS fitter ROIs at known signal and noise positions.  Compute z_amplitude,
total_pe, and chi_sqr for EVERY ROI (no gates applied) and compare their discriminating
power across the full photon-count range.

```
z_amplitude = sum(|pfit[7:10]|) / sqrt(sum(diag(pcov_scaled)[7:10]))
            = (|sqrt(A_B)| + |sqrt(A_G)| + |sqrt(A_R)|) / sqrt(var(sqrt(A_B)) + ...)
```

where `pcov_scaled = pcov_raw * chisqr` (same scaling as `process_covariance`).
This means high chi_sqr *inflates* the covariance → *reduces* z, making the statistic
automatically conservative for poor fits — without a separate chi_sqr gate.

See `claude/covariance_snr_filter.md` for the full implementation plan.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import sys
import time
import types
from scipy.optimize import leastsq


from pyS3M import gaussoptfuncs
from pyS3M import IOFunctions
from pyS3M import Multicolour_Simulation_Functions
from pyS3M import PSFFunctions
from pyS3M import SpectralFunctions
from pyS3M import sCMOSFunctions
from pyS3M import MaskFunctions
from pyS3M.Multicolour_Simulation_Functions import CameraParameters, SimulationConfig

IO  = IOFunctions.IO_Functions()
MSF = Multicolour_Simulation_Functions.MultiC_Sim_Funcs()
PSF = PSFFunctions.PSF_Functions()
S_F = SpectralFunctions.Spectral_Funcs()
sCMOS = sCMOSFunctions.sCMOS_Functions()
M_F = MaskFunctions.Mask_Functions()

print('Imports done.')

In [ ]:
# ── Camera & spectral setup (Ximea medians, same as testing_initial_guess_fit.ipynb) ──
image_size = 14
masks = M_F.get_masks(size_x=image_size, size_y=image_size)
masks_3d = np.dstack([masks[c] for c in ['B', 'G', 'R']])

R, G, B, wavelength = S_F.getpixelefficiency()
pixel_QYs = np.vstack([B, G, R])

data_folder = '../../Camera_Calibrations/Ximea_Camera/'
gain      = IO.read_tiff(os.path.join(data_folder, 'gain.tif'))
offset    = IO.read_tiff(os.path.join(data_folder, 'offset.tif'))
variance  = IO.read_tiff(os.path.join(data_folder, 'variance.tif'))
readnoise = IO.read_tiff(os.path.join(data_folder, 'readnoise.tif'))
rqe       = IO.read_tiff(os.path.join(data_folder, 'rqe.tif'))

camera_parameters = {
    'gain':      np.full((image_size, image_size), np.median(gain)),
    'offset':    np.full((image_size, image_size), np.median(offset)),
    'variance':  np.full((image_size, image_size), np.median(variance)),
    'readnoise': np.full((image_size, image_size), np.median(readnoise)),
    'rqe':       np.full((image_size, image_size), np.median(rqe)),
    'masks':     masks,
    'pixel_QYs': pixel_QYs,
    'pixel_order': ['B', 'G', 'R'],
    'pixel_order_indices': {'B': 0, 'G': 1, 'R': 2},
}
camera_params = CameraParameters.validate_and_create(camera_parameters)

smoothing_function = types.SimpleNamespace()
smoothing_function.args = {'sigma': 1.5}
smoothing_function.extent = 1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = 'image'

# ATTO 565 — representative single-colour dye
dye = 'ATTO 565'
notch_filter   = 'semrock-nf03-405-488-561-635e'
dichroic_mirror = 'semrock-di03-r405-488-561-635-t1-25x36'
filters = [dichroic_mirror, notch_filter]

average_emission_wavelength, dye_pixel_efficiency = (
    MSF.spectral.get_pixel_fractions_dye_and_filters(
        [dye], filters, wavelength, pixel_QYs, normalized=False
    )
)

NA         = 1.49
pixel_size = 69  # nm

print(f'Dye: {dye}')
print(f'Emission wavelength: {average_emission_wavelength:.1f} nm')
print(f'Gain (median): {np.median(gain):.3f} ADU/pe')
print(f'Read noise (median): {np.median(readnoise):.2f} pe')

In [ ]:
def fit_and_compute_z(punctum, smoothed, masks, weights):
    """
    Fit a single ROI with WLS; return [z_amplitude, total_pe, chi_sqr].

    z_amplitude is the Wald t-statistic for the three amplitude parameters:
        z = sum(|pfit[7:10]|) / sqrt(sum(diag(pcov_scaled)[7:10]))
    where pcov_scaled = pcov_raw * chisqr  (same as process_covariance in production).

    pfit[7:10] = [sqrt(A_B), sqrt(A_G), sqrt(A_R)]  (sqrt-space amplitudes)
    diag(pcov_scaled)[7:10] = var(sqrt(A_c)) — uncertainty on each sqrt amplitude.

    Returns NaN array on convergence failure.
    Returns z=0 when pcov is unavailable (degenerate, same behaviour as planned impl.).
    """
    size     = int(punctum.shape[0])
    ravelsize = size * size
    n_params  = 10  # STANDARD: x, y, sy, sx, bg_B, bg_G, bg_R, A_B, A_G, A_R

    ig = np.array(gaussoptfuncs.initial_guess(smoothed, punctum, masks), dtype=np.float64)

    try:
        pfit, pcov_raw, infodict, errmsg, success = leastsq(
            gaussoptfuncs.WLS_chi_nobounds,
            x0=ig,
            args=(punctum, masks, weights, size, ravelsize),
            full_output=True,
            ftol=1e-2,
            xtol=1e-2,
        )

        if success not in [1, 2, 3, 4]:
            return np.full(3, np.nan)

        # Reduced chi-squared (same formula as in _perform_wls_fit)
        residuals = infodict['fvec']
        dof       = max(ravelsize - n_params, 1)
        chisqr    = np.dot(residuals, residuals) / dof

        # Scale covariance by chi_sqr — same as process_covariance()
        # High chi_sqr → larger pcov_scaled → smaller z_amplitude (more conservative)
        if pcov_raw is None or ravelsize <= n_params:
            z_amp    = 0.0
            total_pe = float(pfit[7]**2 + pfit[8]**2 + pfit[9]**2)
            return np.array([z_amp, total_pe, chisqr])

        pcov_scaled = pcov_raw * chisqr

        # z_amplitude: Wald t-statistic
        variances = np.diag(pcov_scaled)[7:10]
        if np.any(variances <= 0):
            z_amp = 0.0
        else:
            z_amp = float(np.sum(np.abs(pfit[7:10])) / np.sqrt(np.sum(variances)))

        total_pe = float(pfit[7]**2 + pfit[8]**2 + pfit[9]**2)
        return np.array([z_amp, total_pe, chisqr])

    except Exception:
        return np.full(3, np.nan)


print('fit_and_compute_z defined.')

In [ ]:
# ── Simulation sweep ──────────────────────────────────────────────────────────────────
# For each photon level we simulate two sets of n_bootstrap ROIs:
#   SIGNAL  — images with a spot at the ROI centre
#   NOISE   — images with no spot (n_photons=0), same camera noise
#
# Photon range spans: noise / dim SM / typical SM / bright (QDot-like)

photon_levels   = [50, 100, 200, 500, 2000, 10000]
n_bootstrap     = 500
background_pe   = 5.0

# Column indices in result arrays
I_Z   = 0  # z_amplitude
I_PE  = 1  # total_pe (A_B + A_G + A_R, fitted)
I_CHI = 2  # chi_sqr

results_signal = {}  # keyed by n_photon
results_noise  = {}  # keyed by n_photon (all 0 pe, same noise level)

image_size_nm = pixel_size * image_size
rng = np.random.default_rng(42)

# Fixed sub-pixel jitter (same across all levels for fair comparison)
x0 = np.full(n_bootstrap, image_size_nm / 2) + rng.uniform(-pixel_size, pixel_size, n_bootstrap)
y0 = np.full(n_bootstrap, image_size_nm / 2) + rng.uniform(-pixel_size, pixel_size, n_bootstrap)
x0y0_signal = {'dye': np.zeros([n_bootstrap, 2, 1])}
x0y0_signal['dye'][:, 0, 0] = x0
x0y0_signal['dye'][:, 1, 0] = y0
x0y0_noise  = x0y0_signal.copy()  # same positions; no photons

t0 = time.time()

def simulate_and_fit(n_photon, x0y0):
    """Simulate one batch and return result array (n_bootstrap, 3)."""
    n_photons = {'dye': np.full(n_bootstrap, n_photon)}
    bayer_image, smoothed_image, _ = MSF.gen_camera_image_stack(
        camera_parameters, wavelength,
        average_emission_wavelength, dye_pixel_efficiency,
        n_photons, x0y0,
        smoothing_function=smoothing_function,
        background_photons=background_pe,
        background_colour=[1, 1, 1],
        NA=NA, pixel_size=pixel_size,
        return_normal_image=False,
    )
    pe   = (bayer_image    - camera_params.offset) / camera_params.gain / camera_params.rqe
    sm   = (smoothed_image - camera_params.offset) / camera_params.gain / camera_params.rqe
    wts  = 1.0 / (camera_params.readnoise**2 + np.maximum(sm, 0) / camera_params.gain)

    res = np.full((n_bootstrap, 3), np.nan)
    for j in range(n_bootstrap):
        res[j] = fit_and_compute_z(
            pe[j].astype(np.float32),
            sm[j].astype(np.float32),
            masks_3d,
            wts[j].astype(np.float32),
        )
    return res


# Simulate noise reference once (same noise level for all comparisons)
print('Simulating noise ROIs (0 pe)...')
noise_ref = simulate_and_fit(0, x0y0_noise)
print(f'  z_med={np.nanmedian(noise_ref[:, I_Z]):.2f}, '
      f'chi_med={np.nanmedian(noise_ref[:, I_CHI]):.2f}')

for n_photon in photon_levels:
    sig = simulate_and_fit(n_photon, x0y0_signal)
    results_signal[n_photon] = sig
    results_noise[n_photon]  = noise_ref  # same noise for all signal levels
    valid = ~np.isnan(sig[:, I_Z])
    print(f'  n_photons={n_photon:6d}: n_valid={valid.sum():4d}, '
          f'z_med={np.nanmedian(sig[:, I_Z]):6.2f}, '
          f'chi_med={np.nanmedian(sig[:, I_CHI]):6.3f}, '
          f'pe_med={np.nanmedian(sig[:, I_PE]):8.1f}')

print(f'\nDone in {time.time()-t0:.1f} s')

In [ ]:
# ── Diagnostic plots ─────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
colors = plt.cm.viridis(np.linspace(0, 0.85, len(photon_levels)))

# ── Panel 1: z_amplitude distributions ───────────────────────────────────────────────
ax = axes[0, 0]
noise_z = noise_ref[:, I_Z]
ax.hist(noise_z[~np.isnan(noise_z)].clip(0, 20),
        bins=60, alpha=0.6, color='grey', density=True, label='0 pe (noise)')
for i, n_photon in enumerate(photon_levels):
    d = results_signal[n_photon][:, I_Z]
    ax.hist(d[~np.isnan(d)].clip(0, 30), bins=60, alpha=0.4, density=True,
            color=colors[i], label=f'{n_photon} pe')
ax.axvline(3.0, color='red', ls='--', lw=2, label='z = 3 threshold')
ax.set_xlabel('z_amplitude')
ax.set_ylabel('Density')
ax.set_title('z_amplitude: noise vs signal')
ax.legend(fontsize=7, ncol=2)

# ── Panel 2: chi_sqr distributions ───────────────────────────────────────────────────
ax = axes[0, 1]
noise_chi = noise_ref[:, I_CHI]
ax.hist(noise_chi[~np.isnan(noise_chi)].clip(0, 8),
        bins=60, alpha=0.6, color='grey', density=True, label='0 pe (noise)')
for i, n_photon in enumerate(photon_levels):
    d = results_signal[n_photon][:, I_CHI]
    ax.hist(d[~np.isnan(d)].clip(0, 8), bins=60, alpha=0.4, density=True,
            color=colors[i], label=f'{n_photon} pe')
ax.axvline(3.0, color='red', ls='--', lw=2, label='chi_sqr = 3 gate')
ax.set_xlabel('chi_sqr')
ax.set_ylabel('Density')
ax.set_title('chi_sqr: note drift at high photon levels')
ax.legend(fontsize=7, ncol=2)

# ── Panel 3: z_amplitude vs chi_sqr scatter ───────────────────────────────────────────
ax = axes[0, 2]
ax.scatter(noise_ref[:, I_CHI], noise_ref[:, I_Z], s=5, alpha=0.3, c='grey', label='noise')
for i, n_photon in enumerate([100, 500, 2000, 10000]):
    if n_photon not in results_signal:
        continue
    d = results_signal[n_photon]
    valid = ~np.isnan(d[:, 0])
    ax.scatter(d[valid, I_CHI].clip(0, 10), d[valid, I_Z].clip(0, 30),
               s=5, alpha=0.3, label=f'{n_photon} pe')
ax.axhline(3.0, color='red', ls='--', lw=1.5, label='z = 3')
ax.axvline(3.0, color='darkred', ls=':', lw=1.5, label='chi_sqr = 3')
ax.set_xlabel('chi_sqr')
ax.set_ylabel('z_amplitude')
ax.set_title('z_amplitude vs chi_sqr')
ax.legend(fontsize=7)
ax.set_xlim(0, 10)
ax.set_ylim(0, 30)

# ── Panel 4: z_amplitude vs total_pe (scale-invariance) ──────────────────────────────
ax = axes[1, 0]
ax.scatter(noise_ref[:, I_PE].clip(0, None), noise_ref[:, I_Z].clip(0, None),
           s=5, alpha=0.3, c='grey', label='noise')
for i, n_photon in enumerate(photon_levels):
    d = results_signal[n_photon]
    valid = ~np.isnan(d[:, 0])
    ax.scatter(d[valid, I_PE].clip(0, None), d[valid, I_Z].clip(0, 40),
               s=5, alpha=0.3, color=colors[i], label=f'{n_photon} pe')
ax.axhline(3.0, color='red', ls='--', lw=1.5, label='z = 3')
ax.axvline(50,  color='k',   ls=':',  lw=1.5, label='50 pe gate')
ax.set_xlabel('Fitted total pe (A_B + A_G + A_R)')
ax.set_ylabel('z_amplitude')
ax.set_title('z vs total_pe: 50 pe gate vs z = 3')
ax.legend(fontsize=7)
ax.set_xscale('symlog', linthresh=10)

# ── Panel 5: ROC curves ───────────────────────────────────────────────────────────────
ax = axes[1, 1]

# Build signal / noise arrays aggregated across all photon levels
all_signal_z   = np.concatenate([results_signal[p][:, I_Z]  for p in photon_levels])
all_signal_pe  = np.concatenate([results_signal[p][:, I_PE] for p in photon_levels])
all_signal_chi = np.concatenate([results_signal[p][:, I_CHI] for p in photon_levels])
all_noise_z    = np.tile(noise_ref[:, I_Z],  len(photon_levels))
all_noise_pe   = np.tile(noise_ref[:, I_PE], len(photon_levels))
all_noise_chi  = np.tile(noise_ref[:, I_CHI], len(photon_levels))

def roc_curve(signal_vals, noise_vals, thresholds, higher_is_signal=True):
    tpr, fpr = [], []
    for t in thresholds:
        if higher_is_signal:
            tpr.append(np.nanmean(signal_vals >= t))
            fpr.append(np.nanmean(noise_vals  >= t))
        else:
            tpr.append(np.nanmean(signal_vals <= t))
            fpr.append(np.nanmean(noise_vals  <= t))
    return np.array(fpr), np.array(tpr)

z_thresh   = np.linspace(0, 20, 200)
pe_thresh  = np.linspace(0, 500, 200)
chi_thresh = np.linspace(0, 10, 200)

fpr_z,   tpr_z   = roc_curve(all_signal_z,   all_noise_z,   z_thresh,   higher_is_signal=True)
fpr_pe,  tpr_pe  = roc_curve(all_signal_pe,  all_noise_pe,  pe_thresh,  higher_is_signal=True)
fpr_chi, tpr_chi = roc_curve(all_signal_chi, all_noise_chi, chi_thresh, higher_is_signal=False)

ax.plot(fpr_z,   tpr_z,   lw=2, label='z_amplitude ≥ threshold')
ax.plot(fpr_pe,  tpr_pe,  lw=2, ls='--', label='total_pe ≥ threshold')
ax.plot(fpr_chi, tpr_chi, lw=2, ls=':',  label='chi_sqr ≤ threshold')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC: signal vs noise (all photon levels)')
ax.legend(fontsize=8)

# ── Panel 6: Summary bar chart ────────────────────────────────────────────────────────
ax = axes[1, 2]
z_threshold  = 3.0
pe_threshold = 50.0

x_pos = np.arange(len(photon_levels))
tpr_z_per_level  = [np.nanmean(results_signal[p][:, I_Z]  >= z_threshold)  for p in photon_levels]
tpr_pe_per_level = [np.nanmean(results_signal[p][:, I_PE] >= pe_threshold) for p in photon_levels]

bar_w = 0.35
ax.bar(x_pos - bar_w/2, tpr_z_per_level,  bar_w, label=f'z ≥ {z_threshold}',  alpha=0.8)
ax.bar(x_pos + bar_w/2, tpr_pe_per_level, bar_w, label=f'pe ≥ {pe_threshold}', alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels([str(p) for p in photon_levels], rotation=45)
ax.set_xlabel('True photons')
ax.set_ylabel('True positive rate')
ax.set_title('TPR per photon level (fixed thresholds)')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=8)

plt.suptitle(
    f'z_amplitude vs current gates — {dye}, bg={background_pe} pe, n={n_bootstrap} per level',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.show()

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────────────
# At z_threshold=3 and pe_threshold=50: FPR, TPR, false-negatives per photon level

z_thr  = 3.0
pe_thr = 50.0
chi_thr = 3.0

noise_z_vals   = noise_ref[:, I_Z]
noise_pe_vals  = noise_ref[:, I_PE]
noise_chi_vals = noise_ref[:, I_CHI]

fpr_z_noise   = np.nanmean(noise_z_vals   >= z_thr)
fpr_pe_noise  = np.nanmean(noise_pe_vals  >= pe_thr)
fpr_chi_noise = np.nanmean(noise_chi_vals <= chi_thr)  # chi <= 3 passes

print('=' * 90)
print(f'  Noise FPR — z≥{z_thr}: {fpr_z_noise:.3f}  |  '
      f'pe≥{pe_thr}: {fpr_pe_noise:.3f}  |  '
      f'chi≤{chi_thr}: {fpr_chi_noise:.3f}')
print('=' * 90)
print(f'  {"True pe":>10s} | {"TPR (z≥3)":>10s} | {"TPR (pe≥50)":>12s} | '
      f'{"TPR (chi≤3)":>12s} | {"z med":>7s} | {"chi med":>8s}')
print('-' * 90)

for n_photon in photon_levels:
    sig = results_signal[n_photon]
    tpr_z   = np.nanmean(sig[:, I_Z]   >= z_thr)
    tpr_pe  = np.nanmean(sig[:, I_PE]  >= pe_thr)
    tpr_chi = np.nanmean(sig[:, I_CHI] <= chi_thr)
    z_med   = np.nanmedian(sig[:, I_Z])
    chi_med = np.nanmedian(sig[:, I_CHI])
    print(f'  {n_photon:10d} | {tpr_z:10.3f} | {tpr_pe:12.3f} | '
          f'{tpr_chi:12.3f} | {z_med:7.2f} | {chi_med:8.3f}')

print('=' * 90)
print()
print('KEY OBSERVATIONS:')
print(f'  1. z_amplitude threshold=3.0 provides consistent signal separation at ALL photon levels')
print(f'  2. pe≥50 pe gate: adequate at low SNR but imprecise threshold (no noise model)')
print(f'  3. chi_sqr≤3 gate: note performance at very bright spots (chi_sqr drifts up)')
print(f'  4. z_amplitude self-corrects: high chi_sqr → inflated pcov → LOWER z (conservative)')
print(f'     This is the OPPOSITE of the current gate which REJECTS high-chi_sqr fits.')